[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/Modulo%204%20-%20NLP/12_RAG_fundamentos.ipynb)


# RAG — fundamentos con un PDF real

En el **11** buscamos en una lista con embeddings. Aquí hacemos **RAG** (*Retrieval-Augmented Generation*) en la parte de **recuperación**:

1. Leer un PDF desde una URL con **`pypdf`** (en memoria, sin guardar archivo).
2. Un **chunk = una página**.
3. Embeddings con Gemini (igual que el 11).
4. Índice **FAISS** en memoria y consultas con preguntas.

Documento: [The State of Enterprise AI — 2025 Report (OpenAI)](https://cdn.openai.com/pdf/7ef17d82-96bf-4dd1-9df2-228f7f377a29/the-state-of-enterprise-ai_2025-report.pdf)


## Setup — instalar librerías

- `google-genai`, `python-dotenv`, `numpy`, `pypdf`, `faiss-cpu`, `requests`


In [ ]:
%pip install -q google-genai python-dotenv numpy pypdf faiss-cpu requests

In [ ]:
import io
import os
import textwrap
from pathlib import Path

import faiss
import numpy as np
import requests
from dotenv import load_dotenv
from google import genai
from pypdf import PdfReader

BASE_DIR = Path(".")

## 1 — ¿Qué es RAG?

**RAG** = recuperar trozos del documento parecidos a la pregunta y (en un sistema completo) pasárselos a un LLM para responder con evidencia.

Hoy solo **encontramos la página** correcta. Los datos viven en variables de Python: `paginas`, `embeddings`, `indice`.


## 2 — Clave API (igual que el notebook 11)

Archivo `.env` en esta carpeta con `GOOGLE_API_KEY=...` ([AI Studio](https://aistudio.google.com/apikey)).


In [ ]:
# Local
# load_dotenv(BASE_DIR / ".env", override=True)

# API_KEY = (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY") or "").strip()
# if not API_KEY:
#     raise ValueError("Falta GOOGLE_API_KEY en .env — https://aistudio.google.com/apikey")

# cliente = genai.Client(api_key=API_KEY)
# MODELO_EMBEDDING = "gemini-embedding-001"
# print(f"Cliente listo. Modelo: {MODELO_EMBEDDING}")

In [ ]:
# Google Colab: descomenta si no usas .env
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

cliente = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODELO_EMBEDDING = "gemini-embedding-001"

print(f"Cliente listo. Modelo: {MODELO_EMBEDDING}")

## 3 — Leer el PDF desde la URL

Descargamos los bytes en memoria y `pypdf` los lee directo (no hace falta guardar el PDF en disco). **Un chunk = una página.**


In [ ]:
URL_PDF = (
    "https://cdn.openai.com/pdf/7ef17d82-96bf-4dd1-9df2-228f7f377a29/"
    "the-state-of-enterprise-ai_2025-report.pdf"
)

resp = requests.get(URL_PDF, timeout=120)
resp.raise_for_status()
reader = PdfReader(io.BytesIO(resp.content))

paginas = []
for i, page in enumerate(reader.pages):
    paginas.append({
        "pagina": i + 1,
        "texto": (page.extract_text() or "").strip(),
    })

print(f"Páginas leídas: {len(paginas)}")
print(paginas[0]["texto"][:250], "...")

## 4 — Ver una página por número

Cambia `NUM_PAGINA` y ejecuta.


In [ ]:
NUM_PAGINA = 2


def mostrar_pagina(n):
    p = paginas[n - 1]
    print(f"=== Página {p['pagina']} / {len(paginas)} ===\n")
    print(textwrap.fill(p["texto"] or "(sin texto)", width=88))


mostrar_pagina(NUM_PAGINA)

## 5 — Embeddings + índice FAISS (todo en memoria)

- `embeddings` — matriz NumPy (una fila por página).
- `indice` — objeto FAISS para buscar vecinos cercanos.
- `paginas_indexadas` — misma lista de páginas, en el mismo orden que las filas del índice.

Usamos `IndexFlatIP`: compara todas las páginas (vale bien con ~24 vectores). Con vectores normalizados, el producto interno ≈ similitud coseno.


In [ ]:
def embed_textos(textos):
    if isinstance(textos, str):
        textos = [textos]
    resp = cliente.models.embed_content(model=MODELO_EMBEDDING, contents=textos)
    return np.array([e.values for e in resp.embeddings], dtype=np.float32)


paginas_indexadas = [p for p in paginas if len(p["texto"]) > 50]
textos = [p["texto"] for p in paginas_indexadas]

print("Calculando embeddings...")
embeddings = embed_textos(textos)
faiss.normalize_L2(embeddings)

indice = faiss.IndexFlatIP(embeddings.shape[1])
indice.add(embeddings)

print(f"embeddings.shape = {embeddings.shape}")
print(f"vectores en índice = {indice.ntotal}")

## 6 — Consultar

Pregunta → embedding → FAISS devuelve las páginas más parecidas. Puedes preguntar en **español** sobre un PDF en **inglés**.


In [ ]:
# pregunta = "¿Cuál es la manera más usada para construir aplicaciones de IA?"
# pregunta = "¿En qué usan más las empresas la API de OpenAI?"
# pregunta = "¿Qué sectores están adoptando más rápido la inteligencia artificial?"
# pregunta = "¿Cuánto tiempo dicen ahorrar los trabajadores cuando usan ChatGPT en el trabajo?"
# pregunta = "¿Para qué están usando las empresas los GPTs personalizados y los Projects?"
pregunta = "¿Qué países están creciendo más rápido en adopción de IA empresarial?"
k = 3

vec_pregunta = embed_textos(pregunta)
faiss.normalize_L2(vec_pregunta)
scores, idxs = indice.search(vec_pregunta, k)

print(f"Pregunta: {pregunta}\n")
for rank, (idx, score) in enumerate(zip(idxs[0], scores[0]), start=1):
    p = paginas_indexadas[idx]
    print("=" * 88)
    print(f"#{rank}  Página {p['pagina']}  —  score = {score:.4f}")
    print(textwrap.fill(p["texto"][:1000], width=88))
    print()

## 7 — Tipos de índices (FAISS y bases vectoriales)

En este cuaderno usamos **`IndexFlatIP`** + vectores **normalizados** (`faiss.normalize_L2`):

- **Flat** = compara la pregunta contra **todos** los vectores (búsqueda exacta, sin atajos).
- **IP** (*inner product*, producto interno) = con vectores unitarios equivale a **similitud coseno**.

Con ~24 páginas es instantáneo. Con **millones** de vectores hace falta otro tipo de índice.

### Familias principales en FAISS

| Índice | Idea | Ventajas | Desventajas | Cuándo usarlo |
|--------|------|----------|-------------|----------------|
| **Flat** (`IndexFlatL2`, `IndexFlatIP`) | Fuerza bruta: revisa todo | Resultado **exacto**; fácil de entender | Lento y pesado si hay millones de vectores | Miles o menos de vectores (como este PDF) |
| **IVF** (`IndexIVFFlat`, …) | Parte el espacio en **clústeres**; solo busca en los más prometedores | Mucho más rápido a gran escala | Aproximado: puede **perder** el vecino correcto; hay que **entrenar** el índice | Cientos de miles / millones de documentos |
| **HNSW** (`IndexHNSWFlat`, …) | Grafo de vecinos: salta de nodo en nodo | Muy rápido en consulta; buen equilibrio calidad/velocidad | Aproximado; más RAM; parámetros que afinar | Producción con muchas consultas por segundo |
| **PQ** (*Product Quantization*) | Comprime cada vector en códigos cortos | Poca memoria en disco/RAM | Pierde precisión; peor ranking fino | Cuando no caben todos los embeddings en RAM |

A menudo se **combinan** (ej. `IVF + PQ`: rápido y compacto).

### L2 vs IP (métrica)

| Métrica | Qué mide | En FAISS |
|---------|----------|----------|
| **L2** | Distancia euclidiana (más lejos = menos parecido) | `IndexFlatL2` |
| **IP / coseno** | Ángulo entre vectores (con normalización ≈ coseno) | `IndexFlatIP` + `normalize_L2` |

Para embeddings de texto suele ir mejor **coseno** (IP + normalizar), que es lo que hicimos.

### ¿Y Pinecone, Chroma, pgvector…?

Son **bases o servicios** que por dentro también usan índices (a veces HNSW u otros). La lógica es la misma:

1. Guardas vectores + metadatos (texto, número de página).
2. Consultas con el embedding de la pregunta.
3. Recuperas los **k** más cercanos.

FAISS es la **librería** (tú montas todo). Chroma/Pinecone te dan API y persistencia listas.

### Regla práctica para elegir

- **Menos de ~10 000 vectores** → `IndexFlatIP` (este notebook).
- **10⁵ – 10⁷** → IVF o HNSW.
- **Poca RAM** → PQ o servicio gestionado.
- **No puedes perder el mejor match** → Flat o HNSW conservador; evita IVF muy agresivo.

**Cierre.** Variables del pipeline: `paginas_indexadas`, `embeddings`, `indice`. Siguiente paso: pasar las páginas recuperadas a un LLM para la respuesta final. Prueba otras preguntas cambiando `pregunta` en la sección 6.
